# CCE PoC v4: Feasibility probe on a contamination-resistant benchmark

**Goal.** Test whether the v3 gating signals (NLI-SE, FLARE, SAPLMA) carry predictive power on a benchmark where the model genuinely lacks the answer without retrieval. The v3 80-question benchmark had 67–85% never-retrieve accuracy across Qwen / DeepSeek / CodeLlama, leaving only 6–16pp headroom for adaptive retrieval to operate within. A reviewer can credibly argue most of the v3 'win' is preserved by skipping retrieval on the easy majority. This probe tests whether moving to a harder, contamination-resistant benchmark restores meaningful headroom.

**Decision criteria for committing to the full study.** All three must hold:
1. **never-retrieve accuracy < 60%** on the new benchmark (vs v3's 72–86%).
2. **always_retrieve − never_retrieve gap > 30pp** (vs v3's 6–16pp).
3. **NLI-SE AUC on `needs_retrieval` ≥ 0.65** with non-trivial positive class (≥ 15 positives in 50 questions).

If all three hold → commit to a 2–3 week full study (3 models × ~200 questions × full ablation grid).
If any fail → diagnose and re-plan before sinking more compute.

**Single-model probe.** Qwen2.5-Coder-7B only (the model where NLI-SE was strongest in v3 — best signal-to-test).

**Wall-clock on A100.** ~50 questions × (1 never-retrieve + 5 SE samples + 1 always-retrieve) generations + NLI clustering ≈ 60–75 min.

**Resumable.** Per-phase artifacts saved to `/content/drive/MyDrive/cce_poc_v4`. Disconnect-safe.

**What this notebook does NOT do.** No multi-model run. No full ablation across all 9 arms. No bootstrap CIs. Those come *after* this probe gives a green light. Cutting that scope is intentional: the probe answers a single yes/no question (does the methodology transfer?) and nothing else.

## Phase 0 — Environment setup

In [ ]:
!pip install -q torch transformers accelerate bitsandbytes sentence-transformers scikit-learn datasets

In [ ]:
# GPU sanity check
import torch
if not torch.cuda.is_available():
    raise SystemError('No GPU detected. Switch runtime → GPU (A100 preferred, L4 fallback).')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'CUDA: {torch.version.cuda}')
print(f'free VRAM: {torch.cuda.mem_get_info()[0] / 1e9:.1f} GB')

In [ ]:
import os
if not os.path.exists('/content/reposynth'):
    !git clone https://github.com/aniJani/reposynth.git /content/reposynth
%cd /content/reposynth
!git checkout Research
!git pull --rebase || true

In [ ]:
from huggingface_hub import login
from google.colab import userdata
try:
    login(userdata.get('HF_TOKEN'))
    print('HF login OK')
except Exception as e:
    print('Set HF_TOKEN as a Colab secret first:', e)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT_DIR = '/content/drive/MyDrive/cce_poc_v4'
!mkdir -p {OUT_DIR}
print(f'will write artifacts to {OUT_DIR}')

## Phase 1 — Load and inspect a contamination-resistant benchmark

We try several published candidates in priority order. Each candidate is a pair `(hf_dataset_name, optional_config)`. We don't pre-commit to a specific one because HuggingFace dataset paths drift; we let the loader find what's available, print the structure of each, then pick.

**Priority order and rationale:**
1. **RepoQA** (`evalplus/repoqa` or similar) — function-search QA over 50 recent repos curated post-cutoff. Output is a function name → easy to verify, easy to NLI-cluster.
2. **CodeRAG-Bench RepoEval** (`code-rag-bench/repoeval` or similar) — API-invocation completion from 8 repos with explicit retrieval/no-retrieval splits. Adapts to QA via 'what call comes next'.
3. **CrossCodeEval** Python (`THUDM/CrossCodeEval` or similar) — cross-file completion. Awkward fit (completion ≠ free-form QA) but published and large.
4. **DS-1000** (`xlangai/DS-1000`) — data-science Python questions; 1000 problems with reference solutions. Free-form-ish, library-knowledge-heavy.

If none load: the cell prints the failure mode for each, and we decide manually. We do *not* fall back to the v3 80-question benchmark — that defeats the entire purpose of this probe.

In [ ]:
from datasets import load_dataset, get_dataset_config_names
import json

CANDIDATES = [
    # (hf_id, config, split) — priority order. Many are guesses; we surface the failures so
    # we can pick a working one if the first attempts miss.
    ('evalplus/repoqa',                     None,           'test'),
    ('evalplus/repoqa-data',                None,           'test'),
    ('bigcode/repoqa',                      None,           'test'),
    ('repoqa/repoqa',                       None,           'test'),
    ('code-rag-bench/repoeval',             None,           'test'),
    ('code-rag-bench/programming-solutions', None,          'test'),
    ('code-rag-bench/programming-solutions', None,          'train'),
    ('THUDM/LongBench',                     'lcc',          'test'),
    ('THUDM/LongBench',                     'repobench-p',  'test'),
    ('tianyang/repobench_python_v1.1',      None,           'test'),
    ('tianyang/repobench-r-cff-python',     None,           'test'),
    ('THUDM/CrossCodeEval',                 'python',       'test'),
    ('microsoft/cceval',                    'python',       'test'),
    ('xlangai/DS-1000',                     None,           'test'),
    ('princeton-nlp/SWE-bench_Lite',        None,           'test'),
]

loaded = {}
errors = {}
for hf_id, config, split in CANDIDATES:
    label = f'{hf_id}' + (f' [{config}]' if config else '') + f' :{split}'
    try:
        if config is None:
            try:
                ds = load_dataset(hf_id, split=split)
            except ValueError as ve:
                if 'Config name is missing' in str(ve):
                    cfgs = get_dataset_config_names(hf_id)
                    print(f'  {label}: needs config, available = {cfgs[:8]}')
                    if cfgs:
                        ds = load_dataset(hf_id, cfgs[0], split=split)
                        label = f'{hf_id} [{cfgs[0]}] :{split}'
                    else:
                        raise
                else:
                    raise
        else:
            ds = load_dataset(hf_id, config, split=split)
        loaded[label] = ds
        print(f'OK   {label}  n={len(ds)}  features={list(ds.features.keys())[:8]}')
    except Exception as e:
        errors[label] = f'{type(e).__name__}: {str(e)[:200]}'

print(f'\n{len(loaded)} candidate(s) loaded, {len(errors)} failed.')
print('\nFailures (so we can diagnose):')
for k, v in errors.items():
    print(f'  {k}\n    {v}')

if not loaded:
    raise RuntimeError('No candidate benchmark loaded. Pick one from the failures above and load it manually before continuing.')


In [ ]:
# Inspect the first loaded candidate's structure so we can adapt it.
label, ds = next(iter(loaded.items()))
print(f'using: {label}')
print(f'n_total: {len(ds)}')
print(f'features: {ds.features}\n')
print('first row (truncated):')
for k, v in ds[0].items():
    s = str(v)
    print(f'  {k}: {s[:300]}{"..." if len(s) > 300 else ""}')

### Adapt the loaded benchmark into our task format

Our pipeline expects each task to be a dict with at minimum:
- `id` — unique int
- `repo` — string (used for retrieval and per-repo grouping)
- `question` — natural-language prompt
- `context` — the long context blob to retrieve from (or `None` if external retrieval over a repo dir)
- `ground_truth` — string with the expected answer (or function name, depending on benchmark)
- `verify` — callable `(answer:str) -> bool` for substring/structural matching

**Field-mapping logic below tries common patterns** (repoqa, repoeval, cceval, ds1000) and exits cleanly if it can't recognize the schema. If it exits, manually edit the `extract_task` function for the schema that loaded.

In [ ]:
# Schema detection + adapter. Edit `extract_task` if the auto-detection misses.
from typing import Optional

def make_substring_verify(ground_truth: str):
    """Lambda factory: case-insensitive substring match. Conservative."""
    needle = ground_truth.strip().lower()
    def verify(answer: str) -> bool:
        return bool(answer) and needle in answer.lower()
    return verify

def make_funcname_verify(func_name: str):
    """Match the function name as a token (avoid partial-substring false positives)."""
    import re
    pat = re.compile(r'\b' + re.escape(func_name) + r'\b')
    def verify(answer: str) -> bool:
        return bool(answer) and pat.search(answer) is not None
    return verify

def extract_task(row: dict, idx: int, schema_hint: str) -> Optional[dict]:
    """Best-effort schema adapter. Returns task dict or None if can't extract."""
    keys = set(row.keys())

    # ---- RepoQA pattern ----
    # Typical fields: 'repo', 'name' (function name = ground truth), 'description' (NL question),
    #                 'code' (full repo dump or single-file context).
    if {'repo', 'name', 'description'} <= keys or {'repo', 'func_name', 'description'} <= keys:
        gt = row.get('name') or row.get('func_name')
        return {
            'id': idx,
            'repo': row.get('repo', f'repoqa_{idx}'),
            'question': f"Find the function described below. Return ONLY its function name (no parens, no body).\n\nDescription:\n{row['description']}",
            'context': row.get('code') or row.get('content') or row.get('full_context'),
            'ground_truth': gt,
            'verify': make_funcname_verify(gt),
            'schema': 'repoqa',
        }

    # ---- CodeRAG-Bench RepoEval pattern ----
    # Fields like 'prompt', 'reference', 'metadata' with 'task_id', 'docs' for retrieval ground truth.
    if {'prompt', 'reference'} <= keys:
        gt = row['reference'].strip()
        return {
            'id': idx,
            'repo': (row.get('metadata') or {}).get('repo', 'repoeval'),
            'question': f"Complete the following code. Return ONLY the next line(s) that should follow:\n\n{row['prompt']}",
            'context': (row.get('metadata') or {}).get('docs') or row.get('docstring') or None,
            'ground_truth': gt,
            'verify': make_substring_verify(gt[:80]),  # match a meaningful prefix
            'schema': 'repoeval',
        }

    # ---- CrossCodeEval pattern ----
    # Fields like 'prompt', 'groundtruth', 'right_context', 'crossfile_context'.
    if {'prompt', 'groundtruth'} <= keys:
        gt = row['groundtruth'].strip()
        return {
            'id': idx,
            'repo': row.get('repository', 'crosscodeeval'),
            'question': f"Complete the following Python code. Return ONLY the next line(s):\n\n{row['prompt']}",
            'context': row.get('crossfile_context') or row.get('right_context'),
            'ground_truth': gt,
            'verify': make_substring_verify(gt[:60]),
            'schema': 'crosscodeeval',
        }

    # ---- DS-1000 pattern ----
    if {'prompt', 'reference_code'} <= keys or {'prompt', 'code_context'} <= keys:
        gt = (row.get('reference_code') or row.get('code_context') or '').strip()
        return {
            'id': idx,
            'repo': row.get('library', 'ds1000'),
            'question': row['prompt'],
            'context': row.get('code_context'),
            'ground_truth': gt,
            'verify': make_substring_verify(gt[:60]) if gt else (lambda a: False),
            'schema': 'ds1000',
        }

    return None

# Try extracting all rows; report success rate.
tasks = []
fail_count = 0
for i, row in enumerate(ds):
    t = extract_task(dict(row), i, label)
    if t is None:
        fail_count += 1
    else:
        tasks.append(t)
    if i >= 200:  # sample first 200 for inspection
        break

print(f'extracted {len(tasks)} / {min(len(ds), 201)} tasks (failed: {fail_count})')
if not tasks:
    print('\nNo schema matched. Inspect the example above and edit extract_task() in this cell.')
    raise RuntimeError('schema adapter needs manual edits')
print(f'sample task: {{')
for k, v in tasks[0].items():
    if k == 'verify':
        print(f'  {k}: <fn>')
    else:
        s = str(v); print(f'  {k}: {s[:200]}{"..." if len(s) > 200 else ""}')
print('}')

In [ ]:
# Select a 50-question probe subset. Stratify by repo if possible to avoid one-repo dominance.
import random
random.seed(42)

TARGET_N = 50
by_repo = {}
for t in tasks:
    by_repo.setdefault(t['repo'], []).append(t)

n_repos = len(by_repo)
per_repo = max(1, TARGET_N // n_repos) if n_repos else TARGET_N
subset = []
for repo, tlist in by_repo.items():
    random.shuffle(tlist)
    subset.extend(tlist[:per_repo])
subset = subset[:TARGET_N]

# Re-id
for i, t in enumerate(subset):
    t['id'] = i + 1

print(f'subset: {len(subset)} tasks across {len(by_repo)} repos')
from collections import Counter
print(f'per-repo counts: {dict(Counter(t["repo"] for t in subset))}')

# Persist (without the verify lambda, which is rebuilt later from ground_truth).
import json as _json
subset_path = f'{OUT_DIR}/subset.json'
with open(subset_path, 'w') as f:
    _json.dump([
        {k: v for k, v in t.items() if k != 'verify'} for t in subset
    ], f, indent=2, default=str)
print(f'saved subset to {subset_path}')

## Phase 2 — Load Qwen2.5-Coder-7B (4-bit NF4)

In [ ]:
import sys
sys.path.insert(0, '/content/reposynth')

from research.paper.cce_poc_v3 import (
    setup_model_and_embedder, probe_layers_for, build_chat_prompt,
    generate_with_features, generate_simple, sample_generations,
)
from research.paper.runner.cce_features import (
    CCEFeatureExtractor, build_partition_indices,
)
from research.paper.runner.semantic_entropy import semantic_entropy

MODEL = 'Qwen/Qwen2.5-Coder-7B-Instruct'
model, tokenizer, embedder = setup_model_and_embedder(MODEL)
probe_layers = probe_layers_for(MODEL)
# CCE feature partition indices are tokenizer-dependent and computed once.
# A fresh extractor is constructed per task in Phase 3.
code_ids, lang_ids = build_partition_indices(tokenizer, partition='lenient')
print(f'model loaded; probe layers = {probe_layers}; |code_ids|={len(code_ids)}, |lang_ids|={len(lang_ids)}')


## Phase 3 — Never-retrieve generation + per-token features

For each of the 50 questions, generate the answer with the model alone (no repo context). Capture per-token features needed for SAPLMA / FLARE / CCE: hidden-state norms at probe layers, vocabulary entropy stats, attention entropy, response length, partitioned H_code / H_lang.

Saves `phase1__qwen.json` to drive after every 10 tasks.

In [ ]:
import time, json, os
PHASE1_PATH = f'{OUT_DIR}/phase1__qwen.json'

if os.path.exists(PHASE1_PATH):
    print(f'loading cached {PHASE1_PATH}')
    with open(PHASE1_PATH) as f:
        phase1 = json.load(f)
    done_ids = set(phase1.keys())
else:
    phase1 = {}
    done_ids = set()

t0 = time.time()
for t in subset:
    qid = str(t['id'])
    if qid in done_ids:
        continue
    prompt = build_chat_prompt(tokenizer, t['repo'], t['question'], context=None)
    # Fresh extractor per task — accumulates per-token CCE observations for this task only.
    cce_extractor = CCEFeatureExtractor(code_ids=code_ids, lang_ids=lang_ids)
    answer, feats = generate_with_features(
        model, tokenizer, prompt, cce_extractor,
        max_new_tokens=250,
    )
    correct = bool(t['verify'](answer))
    phase1[qid] = {
        'qid': t['id'],
        'repo': t['repo'],
        'question': t['question'][:300],
        'answer': answer,
        'correct': correct,
        'features': feats,
    }
    if len(phase1) % 5 == 0 or len(phase1) == len(subset):
        with open(PHASE1_PATH, 'w') as f:
            json.dump(phase1, f, indent=2, default=str)
        print(f'  saved {len(phase1)}/{len(subset)} ({(time.time()-t0)/60:.1f} min)')

n_correct = sum(1 for v in phase1.values() if v['correct'])
print(f'\nPhase 1 done. never-retrieve accuracy = {n_correct}/{len(phase1)} = {n_correct/len(phase1):.3f}')


## Phase 4 — Sampled generations for semantic entropy

For each task, sample N=5 generations at temperature 0.7 (matching v3). These will be clustered by NLI in the next phase to compute semantic entropy.

In [ ]:
PHASE15_PATH = f'{OUT_DIR}/phase15_se_samples__qwen.json'

if os.path.exists(PHASE15_PATH):
    with open(PHASE15_PATH) as f:
        se_samples = json.load(f)
    done_ids = set(se_samples.keys())
    print(f'loaded {len(se_samples)} cached SE samples')
else:
    se_samples = {}
    done_ids = set()

t0 = time.time()
for t in subset:
    qid = str(t['id'])
    if qid in done_ids:
        continue
    prompt = build_chat_prompt(tokenizer, t['repo'], t['question'], context=None)
    samples = sample_generations(model, tokenizer, prompt, n=5,
                                 max_new_tokens=120, temperature=0.7)
    # Embedding-SE as the cheap baseline; NLI-SE comes in Phase 5.
    emb_se = semantic_entropy(samples, embedder, sim_threshold=0.85)
    se_samples[qid] = {
        'qid': t['id'],
        'samples': samples,
        'embedding_se': emb_se,
    }
    if len(se_samples) % 5 == 0 or len(se_samples) == len(subset):
        with open(PHASE15_PATH, 'w') as f:
            json.dump(se_samples, f, indent=2, default=str)
        print(f'  saved {len(se_samples)}/{len(subset)} ({(time.time()-t0)/60:.1f} min)')

print(f'\nPhase 4 done. {len(se_samples)} tasks have SE samples + embedding-SE.')

## Phase 5 — NLI clustering (DeBERTa-large-MNLI, fp32)

Following Kuhn 2023 / Farquhar 2024: cluster the 5 samples per task by bidirectional NLI entailment. Two generations are in the same semantic cluster iff each entails the other. Then semantic entropy = Shannon entropy over cluster sizes.

**fp32 is required** — DeBERTa attention is not fp16-safe (silent NaN failures). Wrap inference in `torch.autocast(enabled=False)`.

In [ ]:
import math
from transformers import AutoTokenizer, AutoModelForSequenceClassification

PHASE2_NLI_PATH = f'{OUT_DIR}/phase2_nli__qwen.json'

nli_tokenizer = AutoTokenizer.from_pretrained('microsoft/deberta-large-mnli')
nli_model = AutoModelForSequenceClassification.from_pretrained(
    'microsoft/deberta-large-mnli'
).float().cuda()  # fp32 explicitly
nli_model.eval()
labels = nli_model.config.id2label  # {0: 'CONTRADICTION', 1: 'NEUTRAL', 2: 'ENTAILMENT'}
ENTAIL_IDX = next(i for i, l in labels.items() if 'entail' in l.lower())
print(f'NLI model loaded. entailment index = {ENTAIL_IDX}')

@torch.no_grad()
def nli_entail(premise: str, hypothesis: str) -> float:
    """P(entailment | premise, hypothesis), fp32 only."""
    if premise.strip() == hypothesis.strip():
        return 1.0
    enc = nli_tokenizer(premise, hypothesis, return_tensors='pt',
                        truncation=True, max_length=512).to('cuda')
    with torch.autocast(device_type='cuda', enabled=False):
        logits = nli_model(**enc).logits
    probs = torch.softmax(logits.float(), dim=-1)
    return float(probs[0, ENTAIL_IDX])

def nli_cluster(samples: list, threshold: float = 0.5):
    """Bidirectional-entailment clustering: two samples merge iff each entails the other."""
    n = len(samples)
    parent = list(range(n))
    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i
    def union(i, j):
        ri, rj = find(i), find(j)
        if ri != rj:
            parent[ri] = rj
    for i in range(n):
        for j in range(i + 1, n):
            ab = nli_entail(samples[i], samples[j])
            ba = nli_entail(samples[j], samples[i])
            if ab >= threshold and ba >= threshold:
                union(i, j)
    clusters = {}
    for i in range(n):
        clusters.setdefault(find(i), []).append(i)
    return list(clusters.values())

def shannon_entropy_bits(sizes):
    total = sum(sizes)
    if total <= 0:
        return 0.0
    h = 0.0
    for s in sizes:
        p = s / total
        if p > 0:
            h -= p * math.log2(p)
    return h

if os.path.exists(PHASE2_NLI_PATH):
    with open(PHASE2_NLI_PATH) as f:
        nli_se = json.load(f)
    print(f'loaded cached {PHASE2_NLI_PATH} ({len(nli_se)} tasks)')
else:
    nli_se = {}

t0 = time.time()
for qid, entry in se_samples.items():
    if qid in nli_se:
        continue
    samples = entry['samples']
    clusters = nli_cluster(samples, threshold=0.5)
    sizes = [len(c) for c in clusters]
    se_bits = shannon_entropy_bits(sizes)
    max_bits = math.log2(len(samples)) if len(samples) > 1 else 1.0
    nli_se[qid] = {
        'qid': entry['qid'],
        'nli_clusters': clusters,
        'nli_n_clusters': len(clusters),
        'nli_largest_frac': max(sizes) / len(samples),
        'nli_se': se_bits,
        'nli_se_norm': se_bits / max_bits if max_bits > 0 else 0.0,
    }
    if len(nli_se) % 5 == 0 or len(nli_se) == len(se_samples):
        with open(PHASE2_NLI_PATH, 'w') as f:
            json.dump(nli_se, f, indent=2)
        print(f'  NLI clustered {len(nli_se)}/{len(se_samples)} ({(time.time()-t0)/60:.1f} min)')

print(f'\nPhase 5 done.')
from collections import Counter
print(f'  n_clusters distribution: {dict(Counter(v["nli_n_clusters"] for v in nli_se.values()))}')


## Phase 6 — Always-retrieve generation (with retrieved repo context)

For each task, build the retrieval context from the task's `context` field (which already holds the relevant code blob from the benchmark). If `context` is too long, retrieve the top-3 chunks closest to the question via the sentence-transformer embedder. Then re-generate.

In [ ]:
from research.paper.cce_poc_v3 import build_chunks_and_index, retrieve

PHASE3_PATH = f'{OUT_DIR}/phase3_always_retrieve__qwen.json'

if os.path.exists(PHASE3_PATH):
    with open(PHASE3_PATH) as f:
        phase3 = json.load(f)
    print(f'loaded {len(phase3)} cached always-retrieve answers')
else:
    phase3 = {}

MAX_CTX_CHARS = 6000  # model context budget for the retrieved-context payload

t0 = time.time()
for t in subset:
    qid = str(t['id'])
    if qid in phase3:
        continue
    ctx_raw = t.get('context') or ''
    if not ctx_raw:
        ctx = None
    elif len(ctx_raw) <= MAX_CTX_CHARS:
        ctx = ctx_raw
    else:
        # retrieve top-3 chunks via embedder
        chunks, embeds = build_chunks_and_index({t['repo']: ctx_raw}, embedder, chunk_size=80)
        retrieved = retrieve(t['question'], chunks, embeds, embedder, top_k=3, target_file=None)
        ctx = '\n\n'.join(retrieved) if isinstance(retrieved, list) else retrieved
    prompt = build_chat_prompt(tokenizer, t['repo'], t['question'], context=ctx)
    answer = generate_simple(model, tokenizer, prompt, max_new_tokens=250)
    correct = bool(t['verify'](answer))
    phase3[qid] = {
        'qid': t['id'],
        'repo': t['repo'],
        'answer': answer,
        'correct': correct,
        'context_chars_used': len(ctx) if ctx else 0,
    }
    if len(phase3) % 5 == 0 or len(phase3) == len(subset):
        with open(PHASE3_PATH, 'w') as f:
            json.dump(phase3, f, indent=2, default=str)
        print(f'  saved {len(phase3)}/{len(subset)} ({(time.time()-t0)/60:.1f} min)')

n_always = sum(1 for v in phase3.values() if v['correct'])
print(f'\nPhase 6 done. always-retrieve accuracy = {n_always}/{len(phase3)} = {n_always/len(phase3):.3f}')

## Phase 7 — Feasibility analysis + verdict

Compute the three decision criteria and print a clear verdict.

In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score

# Align the three phase outputs by qid.
qids = sorted(phase1.keys(), key=int)
init_correct = np.array([phase1[q]['correct'] for q in qids], dtype=bool)
always_correct = np.array([phase3[q]['correct'] for q in qids], dtype=bool)
needs_retrieval = (always_correct & ~init_correct).astype(int)

# Pull NLI-SE scores per task (continuous)
nli_se_scores = np.array([nli_se[q]['nli_se'] for q in qids])
emb_se_scores = np.array([se_samples[q]['embedding_se']['semantic_entropy'] for q in qids])
# Token-entropy proxy: mean cce_mean from phase1 features
flare_scores = np.array([phase1[q]['features'].get('cce_mean', 0.0) for q in qids])

n = len(qids)
init_acc = float(init_correct.mean())
always_acc = float(always_correct.mean())
gap_pp = (always_acc - init_acc) * 100
n_positives = int(needs_retrieval.sum())

# Task category breakdown
helps = int((always_correct & ~init_correct).sum())
knew = int((init_correct & always_correct).sum())
both_fail = int((~init_correct & ~always_correct).sum())
hurts = int((init_correct & ~always_correct).sum())

print('=' * 64)
print(f'V4 Feasibility probe — Qwen2.5-Coder-7B on {label}')
print('=' * 64)
print(f'\nn_questions: {n}')
print(f'\nTask categories:')
print(f'  helps (needs_retrieval=1):   {helps:3d}  ({helps/n:.1%})')
print(f'  already-knew:                {knew:3d}  ({knew/n:.1%})')
print(f'  both-fail:                   {both_fail:3d}  ({both_fail/n:.1%})')
print(f'  retrieval-hurts:             {hurts:3d}  ({hurts/n:.1%})')
print(f'\nKey numbers:')
print(f'  never-retrieve accuracy:    {init_acc:.3f}')
print(f'  always-retrieve accuracy:   {always_acc:.3f}')
print(f'  dynamic range gap:          {gap_pp:+.1f} pp')

# AUCs (if we have positive class)
auc_nli = auc_emb = auc_flare = float('nan')
if n_positives >= 3 and n_positives < n:
    try:
        auc_nli = float(roc_auc_score(needs_retrieval, nli_se_scores))
        auc_emb = float(roc_auc_score(needs_retrieval, emb_se_scores))
        auc_flare = float(roc_auc_score(needs_retrieval, flare_scores))
    except ValueError:
        pass
print(f'\nGating-signal AUC on needs_retrieval (n_pos={n_positives}):')
print(f'  NLI-SE:        {auc_nli:.3f}')
print(f'  embedding-SE:  {auc_emb:.3f}')
print(f'  FLARE (mean):  {auc_flare:.3f}')

print(f'\n--- Decision criteria ---')
c1 = init_acc < 0.60
c2 = gap_pp > 30
c3 = (not np.isnan(auc_nli)) and auc_nli >= 0.65 and n_positives >= 15
print(f'  (1) never-retrieve acc < 0.60:    {init_acc:.3f}    → {"PASS" if c1 else "FAIL"}')
print(f'  (2) gap > 30 pp:                  {gap_pp:+.1f}      → {"PASS" if c2 else "FAIL"}')
print(f'  (3) NLI-SE AUC ≥ 0.65 & ≥ 15 pos: {auc_nli:.3f} / {n_positives}  → {"PASS" if c3 else "FAIL"}')
all_pass = c1 and c2 and c3
print(f'\nVERDICT: {"GREEN — proceed to full study" if all_pass else "RED — re-plan before scaling"}')

## Phase 8 — Save the final probe artifact

In [ ]:
FINAL_PATH = f'{OUT_DIR}/v4_feasibility__qwen.json'

out = {
    'model': MODEL,
    'benchmark_source': label,
    'n_questions': n,
    'subset_size_target': TARGET_N,
    'task_categories': {
        'helps': helps, 'already_knew': knew,
        'both_fail': both_fail, 'retrieval_hurts': hurts,
    },
    'never_retrieve_accuracy': init_acc,
    'always_retrieve_accuracy': always_acc,
    'gap_pp': gap_pp,
    'n_positives': n_positives,
    'auc': {'nli_se': auc_nli, 'embedding_se': auc_emb, 'flare_mean': auc_flare},
    'decision_criteria': {
        'never_retrieve_lt_60': c1,
        'gap_gt_30pp': c2,
        'nli_se_auc_ge_065_and_15pos': c3,
        'all_pass': bool(all_pass),
    },
    'verdict': 'GREEN' if all_pass else 'RED',
}
with open(FINAL_PATH, 'w') as f:
    json.dump(out, f, indent=2)
print(f'wrote {FINAL_PATH}')
print(json.dumps(out, indent=2))

## What to do next

**If verdict = GREEN:**
Download all five artifacts from `/content/drive/MyDrive/cce_poc_v4/` to the repo at `research/paper/v4_results/`. Commit, then plan the full study (3 models × ~200 questions × all 9 ablation arms × bootstrap CIs). Estimated 2–3 weeks of focused work + ~$100 of compute.

**If verdict = RED:**
Inspect *which* criterion failed before pivoting:
- C1 fail (never-retrieve accuracy still high): the benchmark we picked is also contaminated for this model. Try a different candidate from Phase 1 — RepoQA in particular is curated post-cutoff for major models. If RepoQA *is* the one we used, the issue is that Qwen2.5-Coder-7B's training cutoff is later than RepoQA assumes.
- C2 fail (gap < 30pp): retrieval isn't helping. Inspect Phase 6 contexts — they may not contain the answers the model needs. Check that `context` field of each task carries the relevant repo dump.
- C3 fail (NLI-SE AUC < 0.65 or n_pos < 15): with very few positives the AUC is statistical noise; even with enough positives, the signal genuinely doesn't separate retrieval-helps from retrieval-doesn't on this benchmark. Check whether the v3 finding (NLI-SE AUC 0.88 on Qwen) was an artifact of the v3 benchmark's specific question style.

**Either way, post the JSON output above to the repo session — that's how we decide what to fund next.**